# Activity: Campus Café Sales

Data Science Training Intensive · NC State University Libraries, Data Science Services

**The data:** a campus café's register, August 2025 to May 2026, with one row per menu item per day. The café and its sales are made up.

**The question:** Which menu items earn the most money, and does the weather change what people buy?

Part 1 uses the plain Python from the intro session. Part 2 does more with Polars, the library from the next workshop.

This takes about 35 minutes. Run the cells from the top, one at a time.

This is the learner notebook: write the code under each comment. The [filled notebook](https://colab.research.google.com/github/NCSU-Libraries/dsti-python/blob/main/notebooks/activity-cafe-solutions.ipynb) has every answer.

**Using Colab?** Click **Copy to Drive** first.

Activity page: [https://ncsu-libraries.github.io/dsti-python/activities/cafe.html](https://ncsu-libraries.github.io/dsti-python/activities/cafe.html)

## Get the data

One file holds the data: `cafe_sales.csv`. Each row is one menu item on one day.

- `date`, `item` and `category`, such as Coffee or Bakery
- `units_sold` and `unit_price`, in dollars
- `high_temp_f`: the campus high that day, in °F

The café, the menu and the sales are made up. They come from the Libraries' [Agentic Coding with Claude](https://ncsu-libraries.github.io/agentic-coding-with-claude/) workshop (CC BY 4.0). The file has problems in it, on purpose.

In [ ]:
# Download the data file into this folder (skipped if it's already here)
from pathlib import Path
from urllib.request import urlretrieve

BASE = "https://ncsu-libraries.github.io/dsti-python/data/"
name = "cafe_sales.csv"
if not Path(name).exists():
    urlretrieve(BASE + name, name)
    print("Downloaded", name)

# Part 1: Plain Python

## 1. Read the file

The `csv` module reads a CSV file into a list of dictionaries. There is one dictionary per row, with the column names as keys.

`with` opens the file and closes it again when the indented block ends. (`newline=""` and `encoding="utf-8"` are settings the `csv` module likes.)

In [ ]:
import csv

with open("cafe_sales.csv", newline="", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

In [ ]:
# How many rows?

In [ ]:
# Look at the first row

Every value is text, even `units_sold` (`'115'`) and `unit_price` (`'2.25'`). The `csv` module doesn't guess types.

## 2. A revenue function

Revenue is units sold times unit price. Both are text, so turn them into numbers first. `int()` turns text into a whole number, and `float()` turns text into a number with a decimal point (a float). So `int('115')` gives `115`, and `float('2.25')` gives `2.25`.

In [ ]:
# Write revenue(row): units_sold times unit_price, as numbers

In [ ]:
# Try it on the first row

## 3. Total revenue by item

A dictionary can keep running totals. The key is the item, and the value is its total so far. It's the same pattern as counting: if the item is already a key, add this row's revenue to its total. If it isn't, start a new total. `item in totals` is True when the item is already a key. A `for` loop over a dictionary gives its keys.

`round(total, 2)` keeps two decimal places, which is cents.

In [ ]:
# Build totals: each item's total revenue for the whole year
# Hint: start with totals = {}, loop over rows, and use if/else

# Then print each item and its total, rounded to 2 decimal places

Two things look off. "Cold Brew" and "Cold Brew Coffee" sound like the same drink. And is Latte really that far ahead?

## 4. Find the odd price

Look for a price that doesn't fit. `unit_price` is text, so use `float()` to compare it with a number.

In [ ]:
# Print every row whose unit_price is over $10

A latte for \$45 is a typo for \$4.50. It adds \$3,564 of revenue that never happened. Note it: Part 2 fixes it.

That was a lot of looping. Polars works on a whole column at once.

# Part 2: Polars

## 5. Load the data with Polars

Polars is a library for tables of data. If you haven't installed it yet:

- **In `dss-python`:** run `uv add polars` in a terminal. JupyterLab has one: **File → New → Terminal**. Then choose **Kernel → Restart Kernel…** so the notebook can find Polars. Run the Get the data cell again only if the file is missing. Part 2 doesn't use anything from Part 1, so you can start again from here.
- **With conda:** run `conda install -c conda-forge --override-channels polars`.
- **In Colab:** Polars is already there.

In [ ]:
import polars as pl

`pl.read_csv` reads the whole file into a DataFrame, which is a table. `try_parse_dates=True` turns the `date` text into real dates.

In [ ]:
# Read cafe_sales.csv into df, with dates parsed, and show its shape

In [ ]:
# Look at the first rows

## 6. Fix the two problems

`with_columns` changes columns, and `pl.col("unit_price")` picks one.

`pl.when(...).then(...).otherwise(...)` is an `if`/`else` for a column. `&` means and, so the first fix only touches a Latte priced at 45.0. Every other price stays as it was. `.alias("unit_price")` names the result, so it replaces the old column.

`replace` takes a dictionary of old values and new values.

In [ ]:
# Fix the $45 latte, and call every "Cold Brew Coffee" "Cold Brew"

In [ ]:
# Check: count the rows for each item with value_counts()

Every item should have 264 rows now, one for each day. There are 10 items, where there were 11.

Some prices also went up on January 7. That's a real change, not a mistake, so leave it.

## 7. Revenue by item

Columns multiply row by row, so `pl.col("units_sold") * pl.col("unit_price")` gives the revenue of every row at once, with no loop. `.alias("revenue")` names the new column, and `.sum()` adds up a column.

`group_by` makes one group for each item, and `agg` summarizes each group. `.round(2)` keeps cents, and `sort` with `descending=True` puts the biggest first.

In [ ]:
# Add a revenue column: units_sold times unit_price

In [ ]:
# Total revenue and units by item, biggest earner first

In [ ]:
# Total revenue for the whole year

Drip Coffee sells the most cups, but Latte earns the most. It's still first, by less than it looked in Part 1. The total is \$300,506.75.

## 8. Closed days

On a closed day, every item sold 0. Count those days, then leave them out of the weather question. A closed day isn't a slow day.

`filter` keeps the rows that match a condition, and `.height` is the number of rows.

In [ ]:
# Units sold per day, then count the days with zero

That's 33 closed days. The café was open on the other 231.

## 9. Does the weather matter?

`is_in` keeps the rows whose value is in a list. Here the list is the open days, and `.implode()` packs the dates into one list for `is_in` to use.

In [ ]:
# Keep only open days
# Hint: the dates in daily that have more than 0 units, then is_in

To sort days into weather groups, `pl.when(...).then(...)` can take several `.when(...)` steps. Polars checks them in order, and `.otherwise(...)` catches the rest. `pl.lit("...")` means this exact text, not a column name. `.alias("weather")` names the new column.

`.mean()` averages each group, and `pl.len()` counts its rows (the days).

In [ ]:
# Average daily units of Iced Tea and Hot Chocolate in each weather group
# Hint: the groups are below 60°F, 60–79°F and 80°F or warmer. Group by item and weather, then sort by both.

What you should see: Hot Chocolate sells about three and a half times as much below 60°F as at 80°F or warmer. Iced Tea goes the other way.

## 10. One chart

To chart by month, group by `pl.col("date").dt.truncate("1mo").alias("month")`, which rounds each date down to the first day of its month and names it `month`. Then `sort("month")` puts the months in order, so the bars match their labels. `strftime("%b %Y")` writes a date as text, like `Aug 2025`, and `.to_list()` turns a Polars column into a list for `plt.bar`.

In [ ]:
import matplotlib.pyplot as plt

# Bar chart of revenue by month
# Hint: group by the month, sum the revenue, sort by month, then plt.bar(labels, heights)

The first and last bars are short because the data starts on August 18 and ends on May 8. December is short because the café was closed for most of the second half of the month.

## Check your numbers

| What | You should see |
|---|---|
| Rows in the file | 2640 |
| Part 1: Latte total, with the \$45 typo | 67811.5 |
| Part 1: Cold Brew and Cold Brew Coffee totals | 19324.75 and 16047.0 |
| Rows for each item after the fixes | 264, for 10 items |
| Closed days | 33 (open: 231) |
| Total revenue | 300506.75 |

The top 3 items by revenue:

| Item | Revenue | Units |
|---|---|---|
| Latte | 64247.5 | 13868 |
| Drip Coffee | 55638.5 | 23351 |
| Cold Brew | 35371.75 | 8113 |

Average units per open day, by weather:

| Item | Weather | Units per day | Days |
|---|---|---|---|
| Hot Chocolate | 60–79°F | 13.4 | 110 |
| Hot Chocolate | 80°F or warmer | 6.4 | 34 |
| Hot Chocolate | below 60°F | 22.1 | 87 |
| Iced Tea | 60–79°F | 21.2 | 110 |
| Iced Tea | 80°F or warmer | 27.3 | 34 |
| Iced Tea | below 60°F | 13.6 | 87 |

## Keep going

- Try revenue by day of week with `pl.col("date").dt.weekday()`.
- Compare fall and spring. Some prices went up on January 7: does revenue per open day change with them?
- The [Exploratory Data Analysis with Polars workshop](https://colab.research.google.com/github/NCSU-Libraries/getting-started-with-python/blob/main/workshops/exploratory-data-analysis/exploratory-data-analysis.ipynb) goes further with the same library.
- The [Polars user guide](https://docs.pola.rs/) explains every method you used.
- Help: [go.ncsu.edu/getdatahelp](https://go.ncsu.edu/getdatahelp)